In [ ]:
from pathlib import Path
import os
import sys

import mne
import numpy as np

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

In [ ]:
from utils.decode_trigger import decode_8bit_trigger, convert_dict_trigger
from utils.validation_data import filter_stationary_epochs
from modules.events import get_events_tms_per_task
from modules.ica import ICAProcessor
from modules.preprocessing import fix_stim_artifact_cubic

In [ ]:
raw = mne.io.read_raw_bdf(r"data/raw/V2.bdf", preload=True)

In [ ]:
raw_data = raw.copy()

In [ ]:
emg_ch_names = ["EMG L", "EMG R"]
eog_ch_names = ["EOG"]
raw_data.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_data.set_channel_types({'EOG':'eog'})

montage = mne.channels.make_standard_montage("standard_1020",head_size='auto')
raw_data.set_montage(montage)

In [ ]:
raw_data.drop_channels(emg_ch_names)

In [ ]:
events, event_id = mne.events_from_annotations(raw_data)
event_id, renamed_dict =  convert_dict_trigger(event_id, decode_8bit_trigger)
raw_data.annotations.rename(renamed_dict)
raw_data.event_id = event_id

In [ ]:
raw_data = fix_stim_artifact_cubic(
    inst=raw_data, 
    events=events,
    event_id=7,
    tmin=-0.005,
    tmax=0.01,
    pre_window=0.005,
    post_window=0.005
)

In [ ]:
raw_data.notch_filter(freqs=[60, 120], method='spectrum_fit', filter_length='auto')
raw_data.filter(l_freq=1.0, h_freq=100.0, method='fir', fir_design='firwin')

In [ ]:
raw_data.resample(500)

In [ ]:
raw_data.plot()

In [ ]:
bad_ch =[]
raw_data.drop_channels(bad_ch)

In [ ]:
events, event_id = mne.events_from_annotations(raw_data)
events_tms, events_id_tms = get_events_tms_per_task(events, event_id)

In [ ]:
epochs = mne.Epochs(
    raw_data,
    events=events_tms,
    event_id=events_id_tms,
    tmin=-2,
    tmax=-0.1,
    preload=True,
    baseline=None,
    reject_by_annotation=False
)

In [ ]:
epochs.plot(block=True)

In [ ]:
bads = []
epochs.drop(bads)

In [ ]:
from autoreject import get_rejection_threshold
reject_thresholds = get_rejection_threshold(
    epochs, 
    ch_types='eeg', 
    decim=2, 
    random_state=42
)

print(f"Limiares ideais calculados: {reject_thresholds}")

In [ ]:
epochs.copy().drop_bad(reject=reject_thresholds)

In [ ]:
ica = ICAProcessor(method="infomax")

In [ ]:
ica.fit(epochs)

In [ ]:
ica.plot_components()

In [ ]:
exclide_comp = ica.get_exclude_components(threshold=0.75)
exclide_comp

In [ ]:
epochs_clean = ica.apply_ica(exclude=exclide_comp)

In [ ]:
epochs_clean.plot(block=True)

In [ ]:
epochs_clean = epochs_clean.interpolate_bads(reset_bads=True)

In [ ]:
epochs_clean.set_eeg_reference('average', projection=False)

In [ ]:
epochs_clean['tms_pulse_task_bilateral'].compute_psd().plot_topomap(vlim=(150, 10000))
epochs_clean['tms_pulse_task_left'].compute_psd().plot_topomap(vlim=(150, 10000))
epochs_clean['tms_pulse_task_right'].compute_psd().plot_topomap(vlim=(150, 10000))

In [ ]:
epochs_clean_validated = filter_stationary_epochs(epochs_clean)

In [ ]:
epochs_clean_validated.save("data/processed/V2/epochs-mvar.fif", overwrite=True)